# Task 4: Model Explainability using Permutation Feature Importance

In this notebook, we will:
1. Understand what Permutation Feature Importance is and why it matters
2. Implement Permutation Feature Importance with K-fold cross-validation
3. Apply it to both Beat Holdout and Patient Holdout models
4. Visualize which ECG features are most important for classification
5. Interpret the results in the context of cardiac physiology

## Why Explainability Matters:

**Black-box models** (like SVM) can achieve high accuracy, but we need to understand:
- **What features** does the model use to make decisions?
- **Are these features clinically meaningful?**
- **Can we trust the model's reasoning?**
- **Which parts of the ECG waveform matter most?**

This is critical for:
- Medical validation and regulatory approval
- Clinician trust and adoption
- Debugging and improving models
- Scientific understanding

## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set(style='whitegrid', palette='muted', font_scale=1.2)
plt.rcParams['figure.figsize'] = (16, 10)

## 2. What is Permutation Feature Importance?

### The Core Idea:

```
1. Train model on original data → Get baseline performance (e.g., 95% accuracy)

2. For each feature (time point in ECG):
   a. SHUFFLE (permute) that feature's values randomly
   b. Make predictions with shuffled feature
   c. Calculate new performance (e.g., 85% accuracy)
   d. Importance = Baseline - New Performance = 95% - 85% = 10%

3. Features with HIGH importance:
   → Shuffling them causes big performance drop
   → Model relies on them heavily
   → IMPORTANT for classification!

4. Features with LOW importance:
   → Shuffling them barely affects performance
   → Model doesn't use them much
   → Less important for classification
```

### Why Permutation?

**Permutation (shuffling) breaks the relationship** between feature and label:
- Original: Feature X → correctly predicts label Y
- After shuffle: Feature X → random noise, can't predict Y
- If performance drops → Feature X was important!

### Slicing Strategy:

For ECG data with **275 time points**, we divide into **11 slices** of **25 samples each**:

```
Slice 0:  Samples 0-24    (early part of heartbeat)
Slice 1:  Samples 25-49
Slice 2:  Samples 50-74
...
Slice 5:  Samples 125-149 (R-peak region - QRS complex)
...
Slice 10: Samples 250-274 (late part of heartbeat)
```

This corresponds to different parts of the cardiac cycle:
- **P-wave**: Atrial depolarization
- **QRS complex**: Ventricular depolarization (R-peak)
- **T-wave**: Ventricular repolarization

## 3. Load Data

We'll apply permutation feature importance to both validation methods.

In [ ]:
# Choose which validation method to analyze
# Options: 'beat' or 'patient'
VALIDATION_METHOD = 'beat'  # Change to 'patient' for patient holdout analysis

print(f"Analyzing: {VALIDATION_METHOD.upper()} HOLDOUT validation\n")

if VALIDATION_METHOD == 'beat':
    train_data = np.loadtxt('train_beats.csv', delimiter=',')
    test_data = np.loadtxt('test_beats.csv', delimiter=',')
    print("Loaded beat holdout data")
elif VALIDATION_METHOD == 'patient':
    train_data = np.loadtxt('train_patients.csv', delimiter=',')
    test_data = np.loadtxt('test_patients.csv', delimiter=',')
    print("Loaded patient holdout data")
else:
    raise ValueError("VALIDATION_METHOD must be 'beat' or 'patient'")

print(f"Training data shape: {train_data.shape}")
print(f"Testing data shape: {test_data.shape}")

## 4. Prepare Data

In [ ]:
# Extract features and labels
X_train = train_data[:, :-2]
y_train = train_data[:, -2].astype(int)

X_test = test_data[:, :-2]
y_test = test_data[:, -2].astype(int)

print(f"Feature dimensions: {X_train.shape[1]} time points")
print(f"Number of training samples: {len(X_train)}")
print(f"Number of testing samples: {len(X_test)}")
print(f"Number of classes: {len(np.unique(y_train))}")

## 5. Define Slicing Parameters

We'll divide the 275 ECG features into 11 slices of 25 samples each.

In [ ]:
# Slicing parameters
NUM_SLICES = 11
SAMPLES_PER_SLICE = 25
TOTAL_FEATURES = NUM_SLICES * SAMPLES_PER_SLICE  # Should be 275

print(f"Slicing configuration:")
print(f"  Number of slices: {NUM_SLICES}")
print(f"  Samples per slice: {SAMPLES_PER_SLICE}")
print(f"  Total features covered: {TOTAL_FEATURES}")
print(f"\nSlice boundaries:")

for i in range(NUM_SLICES):
    start = i * SAMPLES_PER_SLICE
    end = (i + 1) * SAMPLES_PER_SLICE - 1
    print(f"  Slice {i:2d}: Features {start:3d} - {end:3d}")

## 6. Implement Permutation Feature Importance with K-Fold CV

### Algorithm:

```python
For each fold k in K-fold cross-validation:
    1. Split data into train_k and test_k
    2. Train SVM on train_k
    3. Get baseline performance on test_k
    
    For each slice j:
        4. Permute features in slice j of test_k
        5. Get performance on permuted test_k
        6. Importance[j, k] = baseline - permuted_performance
    
Final Importance[j] = Average over all folds
```

In [ ]:
def permutation_feature_importance_cv(X, y, n_slices=11, samples_per_slice=25, 
                                     n_folds=5, metric='f1_macro', random_state=42):
    """
    Compute permutation feature importance using K-fold cross-validation.
    
    Parameters:
    -----------
    X : array-like, shape (n_samples, n_features)
        Feature matrix
    y : array-like, shape (n_samples,)
        Target labels
    n_slices : int
        Number of feature slices
    samples_per_slice : int
        Number of features per slice
    n_folds : int
        Number of cross-validation folds
    metric : str
        Performance metric ('f1_macro' or 'accuracy')
    random_state : int
        Random seed for reproducibility
    
    Returns:
    --------
    importance_scores : array, shape (n_slices,)
        Mean importance score for each slice
    importance_matrix : array, shape (n_slices, n_folds)
        Importance scores for each slice in each fold
    baseline_scores : array, shape (n_folds,)
        Baseline performance in each fold
    """
    
    print(f"\nRunning Permutation Feature Importance with {n_folds}-Fold CV...")
    print(f"Metric: {metric}")
    print(f"This will take several minutes...\n")
    
    # Initialize storage
    importance_matrix = np.zeros((n_slices, n_folds))
    baseline_scores = np.zeros(n_folds)
    
    # Setup K-fold cross-validation (stratified to maintain class balance)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    
    # Iterate through folds
    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f"Processing Fold {fold_idx + 1}/{n_folds}...")
        
        # Split data
        X_train_fold = X[train_idx]
        y_train_fold = y[train_idx]
        X_test_fold = X[test_idx]
        y_test_fold = y[test_idx]
        
        # Standardize features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_fold)
        X_test_scaled = scaler.transform(X_test_fold)
        
        # Train SVM
        svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=random_state)
        svm.fit(X_train_scaled, y_train_fold)
        
        # Get baseline predictions
        y_pred_baseline = svm.predict(X_test_scaled)
        
        # Calculate baseline performance
        if metric == 'f1_macro':
            baseline_score = f1_score(y_test_fold, y_pred_baseline, average='macro', zero_division=0)
        elif metric == 'accuracy':
            baseline_score = accuracy_score(y_test_fold, y_pred_baseline)
        else:
            raise ValueError(f"Unknown metric: {metric}")
        
        baseline_scores[fold_idx] = baseline_score
        print(f"  Baseline {metric}: {baseline_score:.4f}")
        
        # Permute each slice and calculate importance
        for slice_idx in range(n_slices):
            # Define slice boundaries
            start_idx = slice_idx * samples_per_slice
            end_idx = (slice_idx + 1) * samples_per_slice
            
            # Create copy of test data
            X_test_permuted = X_test_scaled.copy()
            
            # Permute the slice (shuffle within the slice for each sample independently)
            for sample_idx in range(X_test_permuted.shape[0]):
                X_test_permuted[sample_idx, start_idx:end_idx] = np.random.permutation(
                    X_test_permuted[sample_idx, start_idx:end_idx]
                )
            
            # Get predictions with permuted feature
            y_pred_permuted = svm.predict(X_test_permuted)
            
            # Calculate performance with permuted feature
            if metric == 'f1_macro':
                permuted_score = f1_score(y_test_fold, y_pred_permuted, average='macro', zero_division=0)
            else:
                permuted_score = accuracy_score(y_test_fold, y_pred_permuted)
            
            # Importance = drop in performance
            importance = baseline_score - permuted_score
            importance_matrix[slice_idx, fold_idx] = importance
        
        print(f"  Fold {fold_idx + 1} complete.")
    
    # Calculate mean importance across folds
    importance_scores = importance_matrix.mean(axis=1)
    
    print("\n" + "="*60)
    print("Permutation Feature Importance Complete!")
    print("="*60)
    print(f"\nMean baseline {metric}: {baseline_scores.mean():.4f} (±{baseline_scores.std():.4f})")
    
    return importance_scores, importance_matrix, baseline_scores

## 7. Run Permutation Feature Importance

**Note**: This will take several minutes to complete as it trains 5 SVM models and tests each with 11 permutations.

In [ ]:
# Run permutation feature importance
importance_scores, importance_matrix, baseline_scores = permutation_feature_importance_cv(
    X=X_train,
    y=y_train,
    n_slices=NUM_SLICES,
    samples_per_slice=SAMPLES_PER_SLICE,
    n_folds=5,
    metric='f1_macro',
    random_state=42
)

## 8. Analyze and Display Results

In [ ]:
print("\n" + "="*70)
print("PERMUTATION FEATURE IMPORTANCE RESULTS")
print("="*70)
print("\nImportance Scores by Slice:")
print(f"{'Slice':<8} {'Time Points':<15} {'Importance':<12} {'Rank'}")
print("-" * 70)

# Sort slices by importance
sorted_indices = np.argsort(importance_scores)[::-1]

for rank, slice_idx in enumerate(sorted_indices, 1):
    start = slice_idx * SAMPLES_PER_SLICE
    end = (slice_idx + 1) * SAMPLES_PER_SLICE - 1
    importance = importance_scores[slice_idx]
    print(f"Slice {slice_idx:<2}  {start:3d} - {end:3d}     {importance:>8.6f}      {rank}")

print("\n" + "="*70)
print(f"Most important slice: Slice {sorted_indices[0]}")
print(f"Least important slice: Slice {sorted_indices[-1]}")
print(f"Importance range: {importance_scores.min():.6f} to {importance_scores.max():.6f}")

## 9. Visualize Feature Importance

In [ ]:
def plot_feature_importance(importance_scores, samples_per_slice=25, figsize=(14, 6)):
    """
    Plot feature importance scores.
    """
    n_slices = len(importance_scores)
    slice_labels = [f"Slice {i}\n({i*samples_per_slice}-{(i+1)*samples_per_slice-1})" 
                   for i in range(n_slices)]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Create bar plot
    colors = plt.cm.RdYlGn_r(importance_scores / importance_scores.max())
    bars = ax.bar(range(n_slices), importance_scores, color=colors, 
                  alpha=0.8, edgecolor='black', linewidth=1.5)
    
    # Add value labels
    for i, (bar, score) in enumerate(zip(bars, importance_scores)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{score:.4f}',
               ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    ax.set_xlabel('ECG Slice (Time Region)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Feature Importance\n(Drop in F1-Score when permuted)', 
                 fontsize=12, fontweight='bold')
    ax.set_title('Permutation Feature Importance by ECG Slice\n' + 
                f'({VALIDATION_METHOD.upper()} Holdout Validation)', 
                fontsize=14, fontweight='bold', pad=20)
    ax.set_xticks(range(n_slices))
    ax.set_xticklabels(slice_labels, rotation=45, ha='right', fontsize=9)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_ylim([0, importance_scores.max() * 1.15])
    
    # Add reference line for mean importance
    mean_importance = importance_scores.mean()
    ax.axhline(mean_importance, color='red', linestyle='--', linewidth=2, 
              label=f'Mean Importance: {mean_importance:.4f}', alpha=0.7)
    ax.legend(loc='upper right', fontsize=10)
    
    plt.tight_layout()
    plt.show()

plot_feature_importance(importance_scores, SAMPLES_PER_SLICE)

## 10. Visualize ECG with Importance Overlay

Show a sample ECG beat with importance scores overlaid to see which parts matter most.

In [ ]:
def plot_ecg_with_importance(X, importance_scores, sample_idx=0, samples_per_slice=25, 
                            figsize=(16, 8)):
    """
    Plot an ECG signal with feature importance overlay.
    """
    # Get a sample ECG beat
    ecg_signal = X[sample_idx]
    n_samples = len(ecg_signal)
    time_points = np.arange(n_samples)
    
    # Create importance color map for each time point
    importance_per_sample = np.zeros(n_samples)
    for slice_idx, importance in enumerate(importance_scores):
        start = slice_idx * samples_per_slice
        end = min((slice_idx + 1) * samples_per_slice, n_samples)
        importance_per_sample[start:end] = importance
    
    # Normalize importance for color mapping
    importance_normalized = importance_per_sample / importance_per_sample.max()
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize, sharex=True, 
                                   gridspec_kw={'height_ratios': [3, 1]})
    
    # Plot ECG signal with color-coded importance
    for i in range(n_samples - 1):
        color = plt.cm.Reds(importance_normalized[i])
        ax1.plot(time_points[i:i+2], ecg_signal[i:i+2], color=color, linewidth=2)
    
    ax1.set_ylabel('ECG Amplitude (Standardized)', fontsize=12, fontweight='bold')
    ax1.set_title('ECG Signal with Permutation Feature Importance Overlay\n' +
                 'Red = High Importance, Light = Low Importance', 
                 fontsize=14, fontweight='bold', pad=15)
    ax1.grid(True, alpha=0.3, linestyle='--')
    
    # Add slice boundaries
    for slice_idx in range(len(importance_scores)):
        boundary = slice_idx * samples_per_slice
        ax1.axvline(boundary, color='gray', linestyle=':', alpha=0.5, linewidth=1)
    
    # Plot importance bar chart aligned with ECG
    slice_centers = [(i + 0.5) * samples_per_slice for i in range(len(importance_scores))]
    colors = plt.cm.Reds(importance_normalized[::samples_per_slice])
    ax2.bar(slice_centers, importance_scores, width=samples_per_slice*0.9, 
           color=colors, alpha=0.7, edgecolor='black')
    
    ax2.set_xlabel('Time Point (Sample Index)', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Importance', fontsize=12, fontweight='bold')
    ax2.set_xlim([0, n_samples])
    ax2.grid(True, alpha=0.3, linestyle='--', axis='y')
    
    # Add slice labels
    for slice_idx in range(len(importance_scores)):
        center = (slice_idx + 0.5) * samples_per_slice
        ax2.text(center, -0.01 * importance_scores.max(), f'S{slice_idx}', 
                ha='center', va='top', fontsize=9, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# Plot first normal beat from training set
normal_indices = np.where(y_train == 1)[0]
if len(normal_indices) > 0:
    plot_ecg_with_importance(X_train, importance_scores, sample_idx=normal_indices[0])

## 11. Variability Analysis Across Folds

Check how consistent the importance scores are across different folds.

In [ ]:
def plot_importance_variability(importance_matrix, samples_per_slice=25, figsize=(14, 8)):
    """
    Plot importance scores across folds with error bars.
    """
    n_slices, n_folds = importance_matrix.shape
    mean_importance = importance_matrix.mean(axis=1)
    std_importance = importance_matrix.std(axis=1)
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot mean with error bars
    x = np.arange(n_slices)
    ax.bar(x, mean_importance, yerr=std_importance, 
          capsize=5, alpha=0.7, color='steelblue', 
          edgecolor='black', linewidth=1.5, 
          error_kw={'linewidth': 2, 'ecolor': 'darkred'})
    
    # Add individual fold points
    for fold_idx in range(n_folds):
        ax.scatter(x, importance_matrix[:, fold_idx], 
                  alpha=0.5, s=50, color='red', 
                  label=f'Fold {fold_idx+1}' if fold_idx < 3 else '')
    
    ax.set_xlabel('ECG Slice', fontsize=12, fontweight='bold')
    ax.set_ylabel('Feature Importance Score', fontsize=12, fontweight='bold')
    ax.set_title('Feature Importance Variability Across Folds\n' +
                'Bars: Mean ± Std, Points: Individual Folds', 
                fontsize=14, fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels([f'Slice {i}\n({i*samples_per_slice}-{(i+1)*samples_per_slice-1})' 
                       for i in range(n_slices)], rotation=45, ha='right', fontsize=9)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.legend(loc='upper right', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print("\n" + "="*70)
    print("Variability Statistics:")
    print("="*70)
    print(f"\n{'Slice':<8} {'Mean':<12} {'Std Dev':<12} {'CV (%)':<12}")
    print("-" * 70)
    for i in range(n_slices):
        cv = (std_importance[i] / mean_importance[i] * 100) if mean_importance[i] > 0 else 0
        print(f"Slice {i:<2}  {mean_importance[i]:>8.6f}   {std_importance[i]:>8.6f}   {cv:>8.2f}")

plot_importance_variability(importance_matrix, SAMPLES_PER_SLICE)

## 12. Clinical Interpretation

### ECG Anatomy Mapping:

```
Typical ECG Beat (R-peak centered at sample ~137):

Slice 0-2:   P-wave region (Atrial depolarization)
Slice 3-4:   PR segment / Early QRS
Slice 5-6:   QRS complex (Ventricular depolarization) - R-PEAK
Slice 7-8:   ST segment / Early T-wave
Slice 9-10:  T-wave (Ventricular repolarization)
```

### Expected Findings:

**High Importance Regions (Slices 5-6):**
- **QRS Complex / R-Peak Region**
- Most discriminative for arrhythmia detection
- Different arrhythmias have characteristic QRS morphologies:
  - LBBBB/RBBBB: Wide, abnormal QRS
  - PVC: Wide QRS, no preceding P-wave
  - Normal: Narrow QRS

**Medium Importance Regions (Slices 3-4, 7-8):**
- **PR Segment and ST Segment**
- Help distinguish between:
  - Atrial vs Ventricular origin
  - Conduction abnormalities

**Lower Importance Regions (Slices 0-2, 9-10):**
- **P-wave and T-wave**
- Less discriminative for the specific arrhythmias in this dataset
- Still clinically relevant but not the primary distinguishing features

### Clinical Validation:

If the model shows **high importance for QRS complex**, this is **clinically meaningful** because:
1. ✓ Cardiologists primarily look at QRS for ventricular arrhythmias
2. ✓ QRS width/morphology is a key diagnostic criterion
3. ✓ Model is learning physiologically relevant features

If the model showed high importance for **baseline noise regions**, this would be **concerning** because:
1. ✗ Not physiologically meaningful
2. ✗ Model might be overfitting to artifacts
3. ✗ Would not generalize to new hospitals/equipment

## 13. Statistical Significance Testing

Test if the importance scores are statistically significantly different from zero.

In [ ]:
from scipy import stats

print("\n" + "="*70)
print("Statistical Significance Analysis (One-Sample t-test)")
print("H0: Importance = 0 (no effect), H1: Importance > 0 (significant effect)")
print("="*70)
print(f"\n{'Slice':<8} {'Mean Imp.':<12} {'t-statistic':<12} {'p-value':<12} {'Significant?'}")
print("-" * 70)

for i in range(NUM_SLICES):
    # One-sample t-test (test if mean is significantly greater than 0)
    t_stat, p_value = stats.ttest_1samp(importance_matrix[i, :], 0)
    
    # One-tailed p-value (we only care if importance > 0)
    p_value_one_tailed = p_value / 2 if t_stat > 0 else 1.0
    
    significant = "***" if p_value_one_tailed < 0.001 else "**" if p_value_one_tailed < 0.01 else "*" if p_value_one_tailed < 0.05 else "ns"
    
    print(f"Slice {i:<2}  {importance_scores[i]:>8.6f}   {t_stat:>8.4f}     {p_value_one_tailed:>8.6f}   {significant}")

print("\nSignificance levels: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant")

## 14. Comparison: Beat Holdout vs Patient Holdout

**To compare**, run this notebook twice:
1. First with `VALIDATION_METHOD = 'beat'`
2. Then with `VALIDATION_METHOD = 'patient'`

### Expected Differences:

| Aspect | Beat Holdout | Patient Holdout |
|--------|--------------|------------------|
| **Overall Importance** | Higher values | Lower values |
| **Pattern** | More uniform | More varied |
| **Clinical Relevance** | May include patient-specific patterns | True generalizable patterns |
| **Interpretation** | Includes memorization | True physiological importance |

### Why Differences Occur:

**Beat Holdout:**
- Model can learn patient-specific patterns
- ALL features (including noise) might show importance
- Higher overall importance scores

**Patient Holdout:**
- Model must learn generalizable patterns only
- Only truly discriminative features show importance
- Lower importance scores, but more clinically meaningful

## 15. Save Results

In [ ]:
import pickle

# Save importance results
results = {
    'importance_scores': importance_scores,
    'importance_matrix': importance_matrix,
    'baseline_scores': baseline_scores,
    'validation_method': VALIDATION_METHOD,
    'num_slices': NUM_SLICES,
    'samples_per_slice': SAMPLES_PER_SLICE
}

filename = f'permutation_importance_{VALIDATION_METHOD}_holdout.pkl'
with open(filename, 'wb') as f:
    pickle.dump(results, f)

print(f"\nResults saved to: {filename}")

# Also save as CSV for easy inspection
df_importance = pd.DataFrame({
    'Slice': range(NUM_SLICES),
    'Start_Index': [i * SAMPLES_PER_SLICE for i in range(NUM_SLICES)],
    'End_Index': [(i + 1) * SAMPLES_PER_SLICE - 1 for i in range(NUM_SLICES)],
    'Mean_Importance': importance_scores,
    'Std_Importance': importance_matrix.std(axis=1)
})

csv_filename = f'permutation_importance_{VALIDATION_METHOD}_holdout.csv'
df_importance.to_csv(csv_filename, index=False)
print(f"Results saved to: {csv_filename}")

print("\nImportance Summary:")
print(df_importance)

## 16. Summary and Key Takeaways

### What We've Accomplished:

1. ✓ **Implemented Permutation Feature Importance** with K-fold cross-validation
2. ✓ **Identified important ECG regions** for arrhythmia classification
3. ✓ **Validated clinical relevance** of learned features
4. ✓ **Quantified uncertainty** through cross-validation
5. ✓ **Statistical testing** of feature importance

### Why This Matters:

**For Model Development:**
- Identifies which features to focus on
- Helps debug unexpected behavior
- Guides feature engineering

**For Clinical Validation:**
- Ensures model learns physiologically meaningful patterns
- Builds trust with clinicians
- Supports regulatory approval (FDA, etc.)

**For Scientific Understanding:**
- Reveals what the model "sees"
- Validates against domain knowledge
- Identifies potential artifacts

### Limitations:

1. **Computational Cost**: Requires training many models
2. **Correlation**: Correlated features may show low importance
3. **Slice Granularity**: 25-sample slices may be too coarse
4. **Single Metric**: Only shows importance for overall classification

### Next Steps:

- **Task 5**: Apply same explainability to different classifiers
- Compare importance patterns across models
- Investigate class-specific feature importance
- Use finer-grained slices for detailed analysis